In [0]:
%pip install -U langchain langchain-community databricks-langchain langchain_chroma pypdf
 
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [0]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# 1. CONFIG
# ============================================================

RESUME_PATH = "/Volumes/dev/bronze/raw/resumes/"

CATALOG = "dev"
SCHEMA = "bronze"
TABLE = f"{CATALOG}.{SCHEMA}.resume_chunks"


# ============================================================
# 2. FIND PDFs
# ============================================================

pdf_files = list(Path(RESUME_PATH).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files")

for pdf in pdf_files:
    print(" -", pdf.name)


# ============================================================
# 3. LOAD PDFs
# ============================================================

documents = []

for pdf_file in pdf_files:

    print(f"\nLoading: {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))

    docs = loader.load()

    for doc in docs:

        doc.metadata["source_file"] = pdf_file.name

        # Temporary candidate identifier.
        # Later we can extract the actual candidate name.
        doc.metadata["candidate_id"] = pdf_file.stem

    documents.extend(docs)


print(f"\nTotal pages loaded: {len(documents)}")


# ============================================================
# 4. CHUNK DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")


# ============================================================
# 5. CREATE DATAFRAME
# ============================================================

from pyspark.sql import Row

rows = []

for i, chunk in enumerate(chunks):

    rows.append(
        Row(
            chunk_id=i,
            candidate_id=chunk.metadata.get("candidate_id"),
            source_file=chunk.metadata.get("source_file"),
            page_number=chunk.metadata.get("page", 0),
            content=chunk.page_content
        )
    )


df = spark.createDataFrame(rows)


# ============================================================
# 6. WRITE TO DELTA TABLE
# ============================================================

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)


print(f"\nDelta table created: {TABLE}")


# ============================================================
# 7. VERIFY
# ============================================================

display(
    spark.table(TABLE)
    .orderBy("candidate_id", "page_number", "chunk_id")
)

In [0]:
%pip install -U databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "dev"
SCHEMA = "bronze"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.resume_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.resume_chunks_index"

AI_SEARCH_ENDPOINT = "vector_db"

# Databricks embedding model
EMBEDDING_MODEL = "databricks-gte-large-en"


# ============================================================
# 1. CREATE AI SEARCH CLIENT
# ============================================================

client = AISearchClient()

print("AI Search client created")

In [0]:
spark.sql(f"""
ALTER TABLE {SOURCE_TABLE}
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

print("Change Data Feed enabled")

In [0]:
index = client.create_delta_sync_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="content",
    embedding_model_endpoint_name=EMBEDDING_MODEL
)

print("AI Search index creation started")
print(index.describe())

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient(disable_notice=True)

AI_SEARCH_ENDPOINT = "vector_db"

INDEX_NAME = "dev.bronze.resume_index"

index = client.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

print(index.describe())

In [0]:
import time

while True:

    status = index.describe()

    print(status["status"])

    if status["status"]["detailed_state"].startswith("ONLINE"):
        print("\n✅ AI Search index is ONLINE")
        break

    print("Waiting for AI Search index...")
    time.sleep(10)

In [0]:
query = "Who has Databricks experience?"

results = index.similarity_search(
    query_text=query,
    columns=[
        "chunk_id",
        "candidate_id",
        "source_file",
        "page_number",
        "content"
    ],
    num_results=5
)

results

In [0]:
for row in results["result"]["data_array"]:

    print("=" * 80)

    print("Candidate:", row[1])
    print("Source:", row[2])
    print("Page:", row[3])

    print("\nContent:")
    print(row[4])

In [0]:
from databricks.ai_search.client import AISearchClient
from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate


# ============================================================
# 1. CONFIGURATION
# ============================================================

AI_SEARCH_ENDPOINT = "vector_db"

INDEX_NAME = "dev.bronze.resume_index"

LLM_MODEL = "databricks-gemma-3-12b"

TOP_K = 5


# ============================================================
# 2. AI SEARCH CLIENT
# ============================================================

client = AISearchClient(
    disable_notice=True
)

index = client.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

print("✅ AI Search index connected")


# ============================================================
# 3. LLM
# ============================================================

llm = ChatDatabricks(
    model=LLM_MODEL
)

print("✅ Gemma connected")


# ============================================================
# 4. HR PROMPT
# ============================================================

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI Resume Screening Assistant for HR professionals.

Answer the user's question using ONLY the provided resume information.

RULES:

- Do not invent information.
- Do not mix information between candidates.
- Always mention the candidate's name when discussing a candidate.
- If multiple candidates match, list each candidate only once.
- Combine information from multiple resume chunks belonging to the same candidate.
- If the information is not available, say:
  "This information is not available in the resumes."
- Keep the answer concise and professional.
- Do not provide your internal reasoning.

RESUME CONTEXT:
{context}

HR QUESTION:
{question}

ANSWER:
"""
)


# ============================================================
# 5. RETRIEVER FUNCTION
# ============================================================

def retrieve_resumes(question, top_k=TOP_K):

    results = index.similarity_search(
        query_text=question,
        columns=[
            "chunk_id",
            "candidate_id",
            "source_file",
            "page_number",
            "content"
        ],
        num_results=top_k
    )

    columns = [
        col["name"]
        for col in results["manifest"]["columns"]
    ]

    rows = results["result"]["data_array"]

    retrieved_docs = []

    for row in rows:

        data = dict(zip(columns, row))

        retrieved_docs.append(data)

    return retrieved_docs


# ============================================================
# 6. CHATBOT
# ============================================================

print("\n" + "=" * 60)
print("🤖 RESUME HR ASSISTANT")
print("=" * 60)

print("\nExamples:")
print("  - Who has Databricks experience?")
print("  - Who has PySpark experience?")
print("  - Which candidates have GenAI experience?")
print("  - Tell me about Prabhakar's experience")
print("  - Compare candidates with Databricks experience")
print("  - Who has more than 5 years of experience?")

print("\nType 'exit' to stop.\n")


while True:

    question = input("You: ")

    if question.lower().strip() == "exit":
        print("\nBot: Goodbye!")
        break

    if not question.strip():
        continue


    # --------------------------------------------------------
    # RETRIEVE
    # --------------------------------------------------------

    retrieved_docs = retrieve_resumes(question)


    # --------------------------------------------------------
    # BUILD CONTEXT
    # --------------------------------------------------------

    context_parts = []

    for doc in retrieved_docs:

        context_parts.append(
            f"""
Candidate: {doc.get('candidate_id')}
Source: {doc.get('source_file')}
Page: {doc.get('page_number')}

Resume Content:
{doc.get('content')}
"""
        )

    context = "\n\n".join(context_parts)


    # --------------------------------------------------------
    # GENERATE ANSWER
    # --------------------------------------------------------

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)


    # --------------------------------------------------------
    # CLEAN RESPONSE
    # --------------------------------------------------------

    if isinstance(response.content, str):

        answer = response.content

    elif isinstance(response.content, list):

        answer_parts = []

        for item in response.content:

            if isinstance(item, dict):

                if item.get("type") == "text":
                    answer_parts.append(
                        item.get("text", "")
                    )

        answer = "\n".join(answer_parts)

    else:

        answer = str(response.content)


    print("\nBot:")
    print(answer.strip())

    print("\n" + "-" * 60 + "\n")